# Inferencia por tecnología
Pruebas chicas de la cadena de punta a punta (cámara → encoder del codec → modelo → decoder) en lo que tenga la máquina: CUDA (RTX 3080 Ti), CPU, ROCm,
Apple Silicon (MPS), DirectML y NPU Vitis AI (laptop Ryzen AI). Todo por la CLI: `python -m src.cli <modelo> bench --cadena --device X`.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# la raíz del repo: sube desde donde esté abierto el notebook hasta encontrar src/common.py
RAIZ = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "src" / "common.py").exists()), None)
if RAIZ is None:
  raise SystemExit("abrí el notebook dentro del repo (en Colab, corré antes la primera celda de notebooks/colab.ipynb)")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src import nbui
from src.toolkit import Toolkit

# scripts/run_notebooks.py pone ML2_NB_LIMITE para correr todo chico; a mano queda en None (sin recorte)
LIMITE = int(os.environ.get("ML2_NB_LIMITE", "0")) or None

## 1. Backends disponibles
La detección sale de `src/infer/backends.py`. Un backend nuevo se registra ahí y aparece en esta tabla.

In [ ]:
DISP = nbui.backends()                  # tabla de backends y lista de los disponibles
DEVICE = nbui.device_inicial(DISP)      # el que usa el repo por defecto (cuda, rocm o mps si hay GPU; si no, cpu)
DEV = nbui.device_kw(DEVICE)
print("uso:", DEVICE)

## 2. Configuración
~200 frames por defecto; `MODELO` es cualquier modelo con `bench` (con `ALEATORIO = True` mide la arquitectura sin entrenar: la latencia no depende de los pesos). En el modo prueba de `scripts/run_notebooks.py` solo imprime los comandos.

In [ ]:
MODELO = "genie"          # gpt, genie, maskgit, convlstm_px, lewm, vjepa2... o codec
RUN = "genie_s"           # la corrida cuya arquitectura se mide
ALEATORIO = True          # pesos sin entrenar (mismo costo)
FRAMES = LIMITE or 200    # frames medidos (--limite)
PASOS = "4"               # solo genie y maskgit: pasos de decodificación
DRY_RUN = bool(os.environ.get("ML2_NB_LIMITE"))    # a mano corre de verdad; el modo prueba solo imprime

## 3. La cadena, en bucle cerrado y a ritmo de cámara
`--cadena` mide las tres columnas del protocolo: modelo solo, cadena cerrada y cadena a 50 ms entre frames. Si la GPU está ocupada, la acción queda en la cola.

In [ ]:
extra = dict(pasos=PASOS) if MODELO in ("genie", "maskgit") else {}
t = nbui.lanzar(MODELO, "bench", dry_run=DRY_RUN, segundo_plano=False, cadena=True, ritmo=50, limite=FRAMES,
                run=RUN, aleatorio=ALEATORIO, **extra, **DEV)

## 4. p50 y p99
Por columna del protocolo y la mediana de cada etapa (encoder, modelo, decoder).

In [ ]:
nbui.latencia_etapas();

El piso de ~20 ms a ritmo de cámara se diagnostica con `--diagnostico` (variantes del temporizador y de la espera):

In [ ]:
t = nbui.lanzar(MODELO, "bench", dry_run=True, cadena=True, diagnostico=True, limite=FRAMES, run=RUN, aleatorio=ALEATORIO, **extra, **DEV)

## 5. Nodos que caen a la CPU con ONNX Runtime
En DirectML y en la NPU, los nodos que el provider no soporta se ejecutan en la CPU. `bench_ort --profile` cuenta los nodos y el tiempo por provider para el encoder y el decoder del codec.

In [ ]:
for parte in ("cpu", "dml", "npu"):
  if parte in DISP:
    nbui.ort_perfil(enc=parte, dec=parte, frames=min(FRAMES, 100), dry_run=DRY_RUN)
nbui.nodos_cpu();

## 6. Un mp4 de lo que predice
`demo` con el mismo modelo y dispositivo: real a la izquierda, predicho a la derecha.

In [ ]:
if not DRY_RUN:
  nbui.imaginar(RUN, aleatorio=ALEATORIO, device=DEVICE, pasos=PASOS if MODELO in ("genie", "maskgit") else None)
else:
  nbui.pendiente("el mp4 de la demo", "con DRY_RUN = False se genera y se muestra acá")